
Script générant un attaque adverse simple : rotation. On teste :

- Tous les angles pour toutes les images pour obtenir une accuracy moyenne du réseaux sur un data set en fonction de l'angle
- Tous les angle pour chaque image pour un label donné pour obtenir les angles maximisant ou minimisant la prediction d'un réseau sur ce label


> ... TODO ... # TODO test with rotation augmentation during training


In [1]:
from retinotopy import *

HOST='obiwan.local'
Running on metal mps
Welcome on macOS-14.4.1-arm64-arm-64bit
On date 2024-04-25, Running learning on host obiwan.local with device mps


# testing each network for different rotations

In [ ]:
%ls -l {data_cache}/*results_test-rotations.json

In [ ]:
%rm {data_cache}/*results_test-rotations.json 

In [ ]:
angles = np.linspace(-180, 180, 24, endpoint=False)

In [ ]:
delta_angle = angles[1] - angles[0]
angles_min = angles - delta_angle/2
angles_max = angles + delta_angle/2

# Loading and testing the networks
for data_set_type in data_set_types:
    
    print(50*'=')
    print(f'{data_set_type=}')
    args = Params()
    args.do_rotation = True
    args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
    args.folders = ['val'] # type of images to use
    print(50*'-')
        
    for model_name, lw in  zip(['resnet101', 'resnet50', 'resnet18'], [4, 2, 1]): #, 
        print(f'{model_name=}')

        for do_polar in [True, False]:
            args.do_polar = do_polar
            print(f'{args.do_polar=}')
            print(50*'.')
            df_filename = f'{data_cache}/{datetag}_{data_set_type}_{model_name}_{do_polar=}_results_test-rotations.json'
            if os.path.isfile(df_filename):
                df_angle = pd.read_json(df_filename)
            else:
                results = np.zeros((0, 3))
                model_filename = f'{data_cache}/{datetag}_{data_set_type}_{model_name}_{do_polar=}.pt'

                if os.path.isfile(model_filename):
                    print(f"Loading pre-trained resnet {model_filename}")
                    model = charge_model(model_name=model_name, model_path=model_filename, do_circular=args.do_polar).to(device).eval()

                    for angle, angle_min, angle_max in zip(angles, angles_min, angles_max):
                        print(f'{angle=}', end='')
                        dataloaders = datasets_transforms(args, angle_min=angle_min, angle_max=angle_max, shuffle=False, verbose=False)
                        with torch.no_grad():
                            for i_image, (images, labels) in enumerate(dataloaders['val']):
                                images, labels = images.to(device), labels.to(device)

                                outputs = model(images)

                                _, preds = torch.max(outputs.data, dim=1)
                                detect = (preds == labels.data).cpu().numpy()

                                results_ = np.vstack([i_image*args.batch_size_val + np.arange(len(detect)), angle*np.ones(len(detect)), detect]).T
                                results = np.vstack((results, results_))
                        print(f', Accuracy={detect.mean():.3f}')
                    df_angle = pd.DataFrame(results, columns=['i_image', 'angle', 'match'])     
                    df_angle.to_json(df_filename)                        
                    print(25*'. ')

    print(50*'=')

# analysis: average accuracy for different rotations

In [ ]:
df_angle['match'].mean()


In [ ]:
results

In [ ]:
df_angle

In [ ]:
for angle in angles:   
    print(f"{angle=}, accuracy={df_angle[df_angle['angle'] == angle]['match'].mean():.3f}")

In [ ]:
df_angle['match'].mean()

In [ ]:
np.hstack((angles, 180))

In [ ]:
linestyles = ['-', '-.', ':']

In [ ]:
data_set_types

In [ ]:
for model_name in  ['resnet50', 'resnet18', 'resnet101', ]:
    print(50*'=')
    print(f'{model_name=}')
    fig, ax = plt.subplots(figsize=(fig_width*phi/3, fig_width/phi/2))
    for data_set_type, ls in zip(data_set_types, linestyles):
        print(f'{data_set_type=}')
        print(50*'-')

        for do_polar, color in zip([True, False], ['b', 'r']):

            df_filename = f'{data_cache}/{datetag}_{data_set_type}_{model_name}_{do_polar=}_results_test-rotations.json'

            if os.path.isfile(df_filename):
                df_angle = pd.read_json(df_filename)

                results = [df_angle[df_angle['angle'] == angle]['match'].mean() for angle in np.hstack((angles, -180.))]
                label = 'Retino' if do_polar else 'Cartesian'
                label += f' on {data_set_type}'
                ax.plot(np.hstack((angles, 180)), results, color=color, ls=ls, label=label)
        
    #ax.hlines(xmin=-185, xmax=185, y=1/2, ls='--', ec='gray')
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_xlim(-180, 180)
    ax.set_ylim(0, 1)

    for angle in [-180, -90, 0, 90, 180]:
        ax.axvline(x=angle, c='k', ls='--', lw=1)
    ax.set_xticks([-180, -90, 0, 90, 180])
    #ax.set_yscale("logit", use_overline=True) #one_half="1/2", 
    #ax.set_yticks([.7, .75, .8])
    ax.set_ylabel('Average Accuracy', font=font)
    ax.set_xlabel('Rotation angle (°)', font=font)
    plt.legend(bbox_to_anchor=(0.8, 1), loc='upper center', fontsize=10, edgecolor='none')
    plt.tight_layout()
    # plt.xticks(font=font)
    # plt.yticks(font=font);
    plt.show()

In [ ]:
for ext in exts:
    # fig.savefig(os.path.join('cached_data', f'attack_rotation_imagenet.{ext}'), **opts_savefig)
    fig.savefig(f'figs/attack_rotation_imagenet.{ext}', **opts_savefig)

# analysis: average accuracy for different rotations

In [ ]:
df_angle

In [ ]:
df_angle[df_angle['angle']==0]['match'].mean()

In [ ]:
i_images = df_angle['i_image'].unique()
df_angle['i_image'].shape, i_images.shape

In [ ]:
i_image = 0

In [ ]:
all_rot_matches = df_angle[df_angle["i_image"] == i_image]["match"]
all_rot_matches.min(), all_rot_matches.max()

In [ ]:
# for i_image in i_images:
#     all_rot_matches = df_angle[df_angle["i_image"] == i_image]["match"]
#     if not(all_rot_matches.max() == all_rot_matches.min()):  
#         print(f'{i_image=}, {all_rot_matches}')

In [ ]:
json_filename = f'{data_cache}/{datetag}_rotation_attack.json'

if os.path.isfile(model_filename):
    print(f"Load pre-computed {json_filename=}")
    df_attack = pd.read_json(json_filename, orient='index')
else:
    df_attack = pd.DataFrame([], columns=['model_name', 'data_set_type', 'do_polar', 'accuracy_before_attack', 'accuracy_after_attack']) 

    for model_name in ['resnet50', 'resnet18', 'resnet101', ]:
        print(50*'=')
        print(f'{model_name=}')
        # fig, ax = plt.subplots(figsize=(fig_width*phi/3, fig_width/phi/2))
        for data_set_type, ls in zip(data_set_types, linestyles):
            print(50*'-')
            print(f'{data_set_type=}')
            print(50*'-')

            for do_polar, color in zip([True, False], ['b', 'r']):

                df_filename = f'{data_cache}/{datetag}_{data_set_type}_{model_name}_{do_polar=}_results_test-rotations.json'

                if os.path.isfile(df_filename):
                    df_angle = pd.read_json(df_filename)
                    accuracy_before_attack = df_angle[df_angle['angle']==0]['match'].mean() 

                    i_images = df_angle['i_image'].unique()
                    score = 0
                    for i_image in i_images:
                        all_rot_matches = df_angle[df_angle["i_image"] == i_image]["match"]
                        if not(all_rot_matches.max() == all_rot_matches.min()):  
                            score += 1 # attack succeeded !
                    accuracy_after_attack = (len(i_images)-score)/len(i_images)
                    print(f'{data_set_type=}, {do_polar=}, {accuracy_before_attack=:.3f} -  {accuracy_after_attack=:.3f}')

                    df_attack.loc[len(df_attack.index)] = [model_name, data_set_type, do_polar, accuracy_before_attack, accuracy_after_attack]

    df_attack.to_json(json_filename, orient='index', indent=2)


In [ ]:

# for model_name in ['resnet50', 'resnet18', 'resnet101', ]:
#     print(50*'=')
#     print(f'{model_name=}')
#     # fig, ax = plt.subplots(figsize=(fig_width*phi/3, fig_width/phi/2))
#     for data_set_type, ls in zip(data_set_types, linestyles):
#         print(50*'-')
#         print(f'{data_set_type=}')
#         print(50*'-')

#         for do_polar, color in zip([True, False], ['b', 'r']):

#             df_filename = f'{data_cache}/{datetag}_{data_set_type}_{model_name}_{do_polar=}_results_test-rotations.json'

#             if os.path.isfile(df_filename):
#                 df_angle = pd.read_json(df_filename)

#                 i_images = df_angle['i_image'].unique()
#                 score = 0
#                 for i_image in i_images:
#                     all_rot_matches = df_angle[df_angle["i_image"] == i_image]["match"]
#                     if not(all_rot_matches.max() == all_rot_matches.min()):  
#                         score += 1 # attack succeeded !
#                 accuracy_after_attack = (len(i_images)-score)/len(i_images)
#                 print(f'{data_set_type=}, {do_polar=}, {accuracy_after_attack=:.3f}')
#                 df_attack.loc[len(df_attack.index)] = [model_name, data_set_type, do_polar, accuracy_after_attack]

    #             results = [df_angle[df_angle['angle'] == angle]['match'].mean() for angle in np.hstack((angles, -180.))]
    #             label = 'Retino' if do_polar else 'Cartesian'
    #             label += f' on {data_set_type}'
    #             ax.plot(np.hstack((angles, 180)), results, color=color, ls=ls, label=label)
        
    # #ax.hlines(xmin=-185, xmax=185, y=1/2, ls='--', ec='gray')
    # ax.tick_params(axis='x', labelsize=14)
    # ax.tick_params(axis='y', labelsize=14)
    # ax.set_xlim(-180, 180)
    # ax.set_ylim(0, 1)

    # for angle in [-180, -90, 0, 90, 180]:
    #     ax.axvline(x=angle, c='k', ls='--', lw=1)
    # ax.set_xticks([-180, -90, 0, 90, 180])
    # #ax.set_yscale("logit", use_overline=True) #one_half="1/2", 
    # #ax.set_yticks([.7, .75, .8])
    # ax.set_ylabel('Average Accuracy', font=font)
    # ax.set_xlabel('Rotation angle (°)', font=font)
    # plt.legend(bbox_to_anchor=(0.8, 1), loc='upper center', fontsize=10, edgecolor='none')
    # plt.tight_layout()
    # # plt.xticks(font=font)
    # # plt.yticks(font=font);
    # plt.show()

In [ ]:
df_attack

In [ ]:
df_attack.loc[len(df_attack.index)] = [model_name, data_set_type, do_polar, accuracy_after_attack]

In [ ]:
 pd.DataFrame({'model_name': model_name, 'data_set_type':data_set_type, 'do_polar':do_polar, 'accuracy_after_attack':accuracy_after_attack})

In [ ]:
{'model_name': model_name, 'data_set_type':data_set_type, 'do_polar':do_polar, 'accuracy_after_attack':accuracy_after_attack}